# Topic Modeling on Stand-up Comedy Subtitles (`.srt` files)

## Input / Output ##

**INPUT**: A folder containing one cleaned `.srt` subtitle file per show

---

### I. Global Topic Modeling

- Read all `.srt` files from the input folder  
- Segment subtitles into larger text blocks (e.g., blocks of X minutes)  
- Concatenate all blocks from all shows to create a single global corpus  
- Run a single BERTopic model on this global corpus  

Output:  
- A global overview of the main topics across the corpus  
- A trained BERTopic model for topic assignment  

---

### II. Topic Assignment Per Show

- Assign each text block to a topic (based on the trained BERTopic model)  
- Save results in a CSV file per show  

Output:  
- One CSV per show, saved in the output folder  


### Comment on block segmentation strategies

I explored two different methods to segment subtitles into time-based blocks suitable for topic modeling:

1. **Two-step method**  
   - Using `extract_subtitles()` for raw extraction  
   - Then `merge_subtitles_by_block()` for grouping into fixed-duration blocks  
   → This approach offers more flexibility for text preprocessing and intermediate cleaning.

2. **Combined method**  
   - Using `build_global_corpus_with_provenance()`  
   → This function performs both extraction and segmentation in one step, while retaining detailed metadata (e.g., timestamps, original SRT indices).

I ultimately chose:
- The **two-step method** for **global modeling** (step I), where modularity and large-scale cleaning are essential.  
- The **combined method** for **per-show topic assignment** (step II), as it provides better traceability and preserves temporal coherence for later visualizations.


## Installations ##

In [1]:
#For a new environment
#!pip install pysrt bertopic sentence-transformers nltk


In [2]:
# NLTK setup note (for reproducibility only)
# If issues occur with missing 'punkt' tokenizer, make sure to install both:
# - 'punkt'
# - 'punkt_tab' (required in some environments)

# Example:
# import nltk
# nltk.download('punkt')
# nltk.download('punkt_tab')
# nltk.data.path = ['/your/custom/nltk_data_path']
# print(nltk.data.find('tokenizers/punkt/english.pickle'))
# from nltk.tokenize import word_tokenize
# print(word_tokenize("This is a test."))



In [ ]:
import os
import re
import pysrt

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import pandas as pd
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic



## Helper Functions for Topic Modeling ##


In [4]:
def extract_subtitles(srt_path):
    """
    Reads an SRT file and returns a list of dicts:
    {
      'spectacle': file name,
      'start_s': float,   # start time in seconds
      'end_s': float,     # end time in seconds
      'text': str         # cleaned raw text
    }
    """
    subs = pysrt.open(srt_path)
    subtitle_data = []
    filename = os.path.basename(srt_path)
    
    for sub in subs:
        # convert to number of seconds
        start_sec = sub.start.hours*3600 + sub.start.minutes*60 + sub.start.seconds + sub.start.milliseconds/1000
        end_sec   = sub.end.hours*3600 + sub.end.minutes*60 + sub.end.seconds + sub.end.milliseconds/1000
        
        # basic text cleaning
        clean_text = re.sub(r'\n', ' ', sub.text)
        clean_text = re.sub(r'[^\w\s]', '', clean_text)  # remove punctuation
        clean_text = clean_text.lower().strip()
        
        subtitle_data.append({
            'spectacle': filename,
            'start_s': start_sec,
            'end_s': end_sec,
            'text': clean_text
        })
    
    return subtitle_data




In [ ]:
def merge_subtitles_by_block(subtitle_data, block_size_s=180):
    """
    Groups subtitles into blocks of a fixed duration (in seconds).

    Returns a list of dicts:
    {
      'spectacle': str,
      'block_start_s': float,
      'block_end_s': float,
      'block_text': str   # concatenation of all subtitles in the block
    }
    """
    # first, sort by show name, then by start time
    subtitle_data = sorted(subtitle_data, key=lambda x: (x['spectacle'], x['start_s']))
    
    merged_blocks = []
    
    current_show = None
    current_block_start = None
    current_block_end = None
    current_block_text = []
    
    for sub in subtitle_data:
        show = sub['spectacle']
        start_s = sub['start_s']
        end_s = sub['end_s']
        text  = sub['text']
        
        # if we switch to a new show, close the current block
        if show != current_show:
            # save the current block if it exists
            if current_show is not None:
                merged_blocks.append({
                    'spectacle': current_show,
                    'block_start_s': current_block_start,
                    'block_end_s': current_block_end,
                    'block_text': ' '.join(current_block_text)
                })
            # reset for new show
            current_show = show
            current_block_start = start_s - (start_s % block_size_s)  # round to block
            current_block_end   = end_s
            current_block_text  = [text]
            continue
        
        # still in the same show
        # compute the block index of the current start time
        block_index_current = int(start_s // block_size_s)
        block_index_in_progress = int(current_block_start // block_size_s) if current_block_start else None
        
        # check if we’re still in the same block or not
        if block_index_current != block_index_in_progress:
            # close the previous block
            merged_blocks.append({
                'spectacle': current_show,
                'block_start_s': current_block_start,
                'block_end_s': current_block_end,
                'block_text': ' '.join(current_block_text)
            })
            # start a new block
            current_block_start = start_s - (start_s % block_size_s)
            current_block_end   = end_s
            current_block_text  = [text]
        else:
            # same block, extend it
            current_block_end = max(current_block_end, end_s)
            current_block_text.append(text)
    
    # don't forget to save the last block
    if current_show is not None and current_block_text:
        merged_blocks.append({
            'spectacle': current_show,
            'block_start_s': current_block_start,
            'block_end_s': current_block_end,
            'block_text': ' '.join(current_block_text)
        })
    
    return merged_blocks


In [5]:
def preprocess_texts(texts, language='english'):
    """
    Removes stopwords and words with length <= 2.
    Returns a list of preprocessed strings.
    """
    stop_words = set(stopwords.words(language))
    preprocessed = []
    
    for txt in texts:
        tokens = word_tokenize(txt)
        filtered_tokens = [
            t for t in tokens
            if t not in stop_words and len(t) > 2
        ]
        preprocessed.append(' '.join(filtered_tokens))
    
    return preprocessed


In [ ]:
from umap import UMAP

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    metric='cosine',
    random_state=42  # <-- stabilizing
)

In [ ]:
def build_global_corpus(srt_folder, block_size_s=180, language='english'):
    """
    1) Reads all SRT files from the folder
    2) Groups subtitles into blocks of 'block_size_s' seconds
    3) Concatenates blocks from all shows
    4) Returns a list of dicts for each block
    """
    all_subtitles = []
    
    for fname in os.listdir(srt_folder):
        if fname.endswith(".srt"):
            srt_path = os.path.join(srt_folder, fname)
            extracted = extract_subtitles(srt_path)
            all_subtitles.extend(extracted)
    
    # merge into blocks
    merged_blocks = merge_subtitles_by_block(all_subtitles, block_size_s=block_size_s)
    
    return merged_blocks


In [6]:
def run_bertopic_on_blocks(merged_blocks, language='english', output_csv="global_topics.csv"):
    """
    1) Preprocesses the text of each block
    2) Runs a single global BERTopic model
    3) Assigns a topic to each block
    4) Saves results to a CSV file
    """
    # extract the text from each block
    block_texts = [block["block_text"] for block in merged_blocks]
    
    # text preprocessing
    preprocessed_texts = preprocess_texts(block_texts, language)
    
    # initialize the embedding model
    embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    
    # create BERTopic model
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        nr_topics='50',
        language=language,
        verbose=True
    )
    
    # fit-transform the entire corpus
    topics, probs = topic_model.fit_transform(preprocessed_texts)

    # save the trained model
    topic_model.save("bertopic_model_stable")
    
    # build results DataFrame
    results_df = pd.DataFrame({
        "spectacle": [block["spectacle"] for block in merged_blocks],
        "block_start_s": [block["block_start_s"] for block in merged_blocks],
        "block_end_s":   [block["block_end_s"]   for block in merged_blocks],
        "block_text":    block_texts,
        "preprocessed":  preprocessed_texts,
        "topic":         topics
    })
    
    # get topic summary
    topic_info = topic_model.get_topic_info()
    
    # save CSV file
    results_df.to_csv(output_csv, index=False, encoding='utf-8')
    
    print(f"Model trained on {len(preprocessed_texts)} blocks.")
    print(f"Number of topics found: {len(topic_info) - 1}")  # topic -1 = outliers
    
    return results_df, topic_model


## Step 1 – Global Topic Modeling and Export

This section trains a single BERTopic model on all stand-up specials combined.
The resulting model and topic assignments are stored for later reuse.



In [ ]:
# For a folder containing subtitle files (.srt) of stand-up shows
srt_folder = ""
output_csv = ""

# 1) Build all blocks from subtitles
merged_blocks = build_global_corpus(srt_folder, block_size_s=180, language='english')

# 2) Run a single global topic modeling
results_df, topic_model = run_bertopic_on_blocks(
    merged_blocks, 
    language='english', 
    output_csv=output_csv
)

# Display a sample to check the results
results_df.head(10)


In [8]:
# get the summary list of all topics
topics_overview = topic_model.get_topic_info()
print(topics_overview)

# export this table to a CSV file (topics_overview.csv)
topics_overview.to_csv("", index=False)



    Topic  Count                                     Name  \
0      -1   1260                  -1_like_dont_know_queer   
1       0   1757                     0_like_know_dont_get   
2       1   1127                1_thank_much_welcome_guys   
3       2    109             2_years_20yearolds_kids_year   
4       3     78           3_drink_drinking_coke_martinis   
5       4     77                4_capita_per_record_label   
6       5     72                   5_eat_pizza_chilis_hut   
7       6     44           6_wife_married_friends_stabbed   
8       7     40           7_ladies_gentlemen_women_woman   
9       8     40                   8_otters_dog_pet_seals   
10      9     40             9_city_minneapolis_town_york   
11     10     34             10_baby_babies_pregnant_1960   
12     11     34            11_tonight_night_evening_date   
13     12     34            12_shirt_fat_kirkland_tucking   
14     13     34              13_dying_died_cop_survivors   
15     14     32        

In [ ]:
# About the outputs so far:

# results_df = DataFrame containing one row per text block (segment) extracted from SRT files
#              Includes: block ID, start_time, end_time, block text, assigned topic label, and topic confidence

# topic_model = the trained BERTopic model object

# topics_overview.csv = summary of the discovered topics, saved as a CSV file

## Step 2 – Topic Assignment Per Show ##

Interesting possibility: Apply the model at a finer (or coarser) temporal granularity than during training.

### 1. Helpers functions

In [9]:
import os
import pysrt

def build_global_corpus_with_provenance(srt_folder, block_size_s=180, language='english'):
    """
    Reads all SRT files in srt_folder and splits each into segments of approximately block_size_s seconds.
    For each segment, it keeps:
      - source_file     : name of the original SRT file
      - segment_index   : segment number within the file
      - start_time      : segment start time (in seconds)
      - end_time        : segment end time (in seconds)
      - text            : concatenated text of all subtitles in the segment
      - timestamp_line  : concatenated original timestamp lines (e.g., "00:00:08,133 --> 00:00:11,219")
      - srt_indices     : concatenated subtitle indices (line numbers)
    """
    corpus = []
    for filename in os.listdir(srt_folder):
        if filename.lower().endswith('.srt'):
            srt_path = os.path.join(srt_folder, filename)
            try:
                subtitles = pysrt.open(srt_path, encoding='utf-8')
            except Exception as e:
                print(f"Error opening {filename}: {e}")
                continue
            
            current_segment_text = []
            current_timestamp_lines = []
            current_srt_indices = []
            segment_start = None
            segment_end = None
            segment_index = 0

            for sub in subtitles:
                # get timestamp line as string (e.g., "00:00:08,133 --> 00:00:11,219")
                ts_line = f"{sub.start} --> {sub.end}"
                srt_idx = sub.index  # subtitle index (line number)

                start_sec = sub.start.ordinal / 1000.0
                end_sec = sub.end.ordinal / 1000.0

                if segment_start is None:
                    segment_start = start_sec

                # if subtitle belongs to the current block of block_size_s seconds
                if start_sec - segment_start < block_size_s:
                    current_segment_text.append(sub.text.strip())
                    current_timestamp_lines.append(ts_line)
                    current_srt_indices.append(str(srt_idx))
                    segment_end = end_sec
                else:
                    # save current segment
                    corpus.append({
                        'source_file': filename,
                        'segment_index': segment_index,
                        'start_time': segment_start,
                        'end_time': segment_end,
                        'text': ' '.join(current_segment_text),
                        'timestamp_line': " | ".join(current_timestamp_lines),
                        'srt_indices': ", ".join(current_srt_indices)
                    })
                    # start new segment with current subtitle
                    segment_index += 1
                    current_segment_text = [sub.text.strip()]
                    current_timestamp_lines = [ts_line]
                    current_srt_indices = [str(srt_idx)]
                    segment_start = start_sec
                    segment_end = end_sec

            # save last segment if it exists
            if current_segment_text:
                corpus.append({
                    'source_file': filename,
                    'segment_index': segment_index,
                    'start_time': segment_start,
                    'end_time': segment_end,
                    'text': ' '.join(current_segment_text),
                    'timestamp_line': " | ".join(current_timestamp_lines),
                    'srt_indices': ", ".join(current_srt_indices)
                })
    return corpus


In [10]:
def apply_model_to_1min_blocks_per_show(segments, topic_model):
    """
    Aggregates segments by show (source_file) and by minute, 
    then applies the BERTopic model on the aggregated text to assign a topic to each 1-minute block.

    Expects each segment to contain at least:
      - source_file, start_time, text, timestamp_line, srt_indices.
      
    Returns a DataFrame with, for each show and minute:
      - source_file, minute, aggregated_text, topic, srt_indices, timestamps.
    """
    df = pd.DataFrame(segments)
    
    # compute the corresponding minute for each segment
    df['minute'] = (df['start_time'] // 60).astype(int)
    
    # aggregation function
    def agg_func(grp):
        aggregated_text = " ".join(grp['text'].tolist())
        srt_indices = ", ".join(grp['srt_indices'].tolist())
        timestamps = " | ".join(grp['timestamp_line'].tolist())
        return pd.Series({
            'aggregated_text': aggregated_text,
            'srt_indices': srt_indices,
            'timestamps': timestamps
        })
    
    # group by 'source_file' and 'minute'
    agg_df = df.groupby(['source_file', 'minute'], as_index=False).apply(agg_func)

    # reset index in case groupby().apply() created a multi-index
    agg_df.reset_index(drop=True, inplace=True)

    # apply the trained BERTopic model to get the topics
    topics, _ = topic_model.transform(agg_df['aggregated_text'].tolist())
    agg_df['topic'] = topics
    
    return agg_df



def export_1min_blocks_per_show_to_csv(segments, topic_model, output_folder):
    """
    Aggregates segments by show and by minute, applies the BERTopic model, 
    and exports a CSV per show with the topic labels.
    """
    agg_df = apply_model_to_1min_blocks_per_show(segments, topic_model)
    for show in agg_df['source_file'].unique():
        show_df = agg_df[agg_df['source_file'] == show].copy()
        show_df.sort_values('minute', inplace=True)
        base_name = os.path.splitext(show)[0]
        output_csv = os.path.join(output_folder, f"{base_name}_1min_topics.csv")
        show_df.to_csv(output_csv, index=False)
        print(f"CSV exported for {show} : {output_csv}")
    return agg_df


### 2. Predicting Topics for Each Minute of Each Show

In [ ]:
# 1. Define the SRT folder and generate timestamped segments with provenance
srt_folder = "cleaned_srt"
merged_blocks = build_global_corpus_with_provenance(srt_folder, block_size_s=60, language='english')

# 2. Load the pre-trained BERTopic model (if needed)
# from bertopic import BERTopic
# topic_model = BERTopic.load("bertopic_model_stable")

# 2. Use the already trained BERTopic model
# (We assume 'topic_model' is already loaded/trained in the notebook.)
# We just apply the transform method to get topics.
topics, _ = topic_model.transform([seg['text'] for seg in merged_blocks])
for seg, t in zip(merged_blocks, topics):
    seg['topic'] = t

# 3. Define the output path for the final CSVs
output_csv_folder = ""
import os
if not os.path.exists(output_csv_folder):
    os.makedirs(output_csv_folder)

# 4. Call the export function to generate one CSV per show
agg_df_per_show = export_1min_blocks_per_show_to_csv(merged_blocks, topic_model, output_csv_folder)


### 3.Visualisations 


In [12]:
topics_per_class = (
    results_df
    .groupby(["topic", "spectacle"])
    .size()
    .reset_index(name="count")
    .sort_values(["topic", "count"], ascending=[True, False])
)

topics_per_class.to_csv("", index=False)




In [ ]:
import plotly.express as px


topic_counts = topics_per_class.groupby("topic")["spectacle"].nunique().reset_index()
topic_counts.columns = ["topic", "n_spectacles"]


fig = px.bar(
    topic_counts,
    x="topic",
    y="n_spectacles",
    title="Nombre de spectacles représentés par topic",
    labels={"topic": "Topic", "n_spectacles": "Nombre de spectacles"},
)

fig.update_xaxes(type='category', tickmode='linear', tick0=0, dtick=1)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

